In [ ]:
from ucimlrepo import fetch_ucirepo
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.preprocessing import StandardScaler


In [2]:
wine_quality = fetch_ucirepo(name='Wine Quality')
wine_quality_data = wine_quality['data']['original']
wine_quality_data

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,color
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


In [ ]:
raw_features = wine_quality_data.drop(columns=['quality']).assign(
    color=wine_quality_data['color'].transform(lambda x: (x == 'red').astype(float))
).values

scaler = StandardScaler()
features = torch.tensor(scaler.fit_transform(raw_features), dtype=torch.float32)

min_quality = wine_quality_data['quality'].min()
num_classes = wine_quality_data['quality'].max() - min_quality + 1
labels = torch.tensor((wine_quality_data['quality'].values - min_quality), dtype=torch.long)

dataset = torch.utils.data.TensorDataset(features, labels)
trainloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)


In [ ]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(12, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input):
        f1 = F.relu(self.fc1(input))
        f2 = F.relu(self.fc2(self.dropout(f1)))
        f3 = F.relu(self.fc3(self.dropout(f2)))
        return self.fc4(f3)

net = Net(num_classes)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)


In [ ]:
for epoch in range(50):
    net.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, batch_labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    net.eval()
    with torch.no_grad():
        correct = (torch.argmax(net(features), dim=1) == labels).sum().item()
        acc = correct / len(labels) * 100
    print(f'[Epoch {epoch + 1:2d}] loss: {running_loss / len(trainloader):.3f}  Accuracy: {acc:.2f}%')

print('Finished Training')


[1,   200] loss: 0.128
Accuracy: 43.65%
[1,   400] loss: 0.125
Accuracy: 43.65%
[1,   600] loss: 0.130
Accuracy: 43.65%
[1,   800] loss: 0.130
Accuracy: 43.65%
[1,  1000] loss: 0.130
Accuracy: 43.65%
[1,  1200] loss: 0.133
Accuracy: 43.65%
[1,  1400] loss: 0.127
Accuracy: 43.65%
[1,  1600] loss: 0.125
Accuracy: 43.65%
[2,   200] loss: 0.123
Accuracy: 43.65%
[2,   400] loss: 0.135
Accuracy: 43.65%
[2,   600] loss: 0.128
Accuracy: 43.65%
[2,   800] loss: 0.126
Accuracy: 43.65%
[2,  1000] loss: 0.131
Accuracy: 43.65%
[2,  1200] loss: 0.129
Accuracy: 43.56%
[2,  1400] loss: 0.127
Accuracy: 34.79%
[2,  1600] loss: 0.129
Accuracy: 43.65%
[3,   200] loss: 0.131
Accuracy: 43.65%
[3,   400] loss: 0.123
Accuracy: 43.65%
[3,   600] loss: 0.126
Accuracy: 43.65%
[3,   800] loss: 0.126
Accuracy: 43.65%
[3,  1000] loss: 0.129
Accuracy: 41.33%
[3,  1200] loss: 0.127
Accuracy: 44.64%
[3,  1400] loss: 0.133
Accuracy: 43.65%
[3,  1600] loss: 0.129
Accuracy: 32.91%
[4,   200] loss: 0.124
Accuracy: 43.65%
